In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# LACS re-referencing

In [ ]:
import sys
MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

from makeshift.reref.lacs import reref_lacs, _BMRB_STATS

import makeshift as ms
import numpy as np
import pandas as pd
import os
import glob
import json
import tqdm
import pickle

_N_STD_OUTLIER = 5

def _bmrb_outlier(row, n_std=_N_STD_OUTLIER):
    """Same BMRB-outlier rule lacs.reref_lacs() applies internally before
    fitting. Duplicated here because reref_lacs() only returns the fitted
    offsets, not a per-row mask -- needed to rebuild reref_mask below."""
    if pd.isna(row['Val']):
        return False
    stats = _BMRB_STATS.get(str(row['Comp_ID']).upper(), {}).get(str(row['Atom_ID']).upper())
    if stats is None:
        return True
    mean, std = stats
    return abs(row['Val'] - mean) > n_std * std


def apply_lacs(df, n_std=_N_STD_OUTLIER):
    """
    Adapter around makeshift.reref.lacs.reref_lacs(), rebuilding the old
    ReRef_v1.reref_lacs_() interface: (df, skip, offset_collect) with Val
    corrected in place and a reref_mask column (True = not a BMRB outlier
    AND that atom's offset was successfully fit).
    """
    df = df.copy()
    df['orig'] = df['Val'].copy()

    offsets, check = reref_lacs(df, n_std=n_std)
    offset_collect = {atom: v for atom, v in offsets.items() if v is not None}
    skip = not bool(offset_collect)

    outlier = df.apply(lambda row: _bmrb_outlier(row, n_std), axis=1)
    fitted = df['Atom_ID'].map(check).fillna(False)
    df['reref_mask'] = (~outlier) & fitted

    df['Val'] = df.apply(
        lambda row: row['Val'] + offset_collect[row['Atom_ID']]
        if (row['reref_mask'] and row['Atom_ID'] in offset_collect) else row['Val'],
        axis=1,
    )

    return df, skip, offset_collect


base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
atom_types = ['H','N','CA','CB', 'C']
result_dict = {atom: {} for atom in atom_types}
result_dict.update({atom + "_mask": {} for atom in atom_types})
result_dict["entry_ID"] = {}
result_dict["sequence"] = {}
result_dict['offset'] = {}

idx = 1
out_of_bounds = set()
mismatch = set()
notfit_refer = set()

three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

def get_cs_seq(cs):
    df_unique = cs.drop_duplicates(subset='Seq_ID')
    cs_id = df_unique['Seq_ID'].astype(int).tolist()
    seq_single_letter = df_unique['Comp_ID'].map(three_to_one)
    sequence = ''.join(seq_single_letter.dropna())
    return sequence, cs_id

with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph_notemp.pkl"), 'rb') as f:
    rawdata = pickle.load(f)

processed = 0
for idx, name in enumerate(tqdm.tqdm(list(rawdata.keys())), start=1):
        folder_name = name.split("_")[0]
        folder_path = os.path.join(data_path, folder_name)

        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        df = rawdata[name]["chemical shifts"]

        key = f"{idx}"

        df['Seq_ID'] = df['Seq_ID'].astype(int)
        df = df.sort_values(by='Seq_ID').reset_index(drop=True)

        cs_sequence, cs_dict = get_cs_seq(df)

        df = df.loc[
            df.Atom_ID.isin(['H', 'HA', 'N', 'CA', 'CB', 'C']) | df.Atom_ID.str.contains('^HA', na=False)
        ]
        df['Atom_ID'] = df['Atom_ID'].replace({'HA2': 'HA', 'HA3': 'HA'})
        df['Val'] = df.groupby(['Seq_ID', 'Atom_ID'])['Val'].transform('mean')
        df = df.drop_duplicates(subset=['Seq_ID', 'Atom_ID'])

        df = df.loc[
            df.Atom_ID.isin(['H', 'N', 'CA', 'CB', 'C'])
        ]
        if not df.empty:
            df, skip, offset_collect = apply_lacs(df)
            if skip:
                notfit_refer.add(name)
                continue

            result_dict["entry_ID"][key] = name
            entities = ms.NMRStarEntry.from_file(str_files[0]).sequences()
            sequence = entities['Polymer_seq_one_letter_code'].values[0]
            result_dict["sequence"][key] = sequence

            for atom in atom_types:
                cs_atom = np.full(len(sequence), None, dtype=object)
                cs_atom_mask = np.full(len(sequence), False, dtype=bool)
                sub_df = df[df['Atom_ID'] == atom]
                for _, row in sub_df.iterrows():
                    seq_idx_cs = int(row['Seq_ID'])
                    reref_mask = row['reref_mask']
                    try:
                        if three_to_one[row["Comp_ID"]] != sequence[seq_idx_cs - 1].upper():
                            print(f"Residue mismatch at Seq_ID {row['Seq_ID']} in {name}")
                            mismatch.add(name)
                        elif reref_mask:
                            cs_atom[seq_idx_cs - 1] = row['Val']
                            # cs_atom stays None / mask stays False when reref_mask is False
                            cs_atom_mask[seq_idx_cs - 1] = reref_mask
                    except IndexError:
                        print(f"Seq_ID {row['Seq_ID']} out of bounds in {name}")
                        out_of_bounds.add(name)

                result_dict[atom][key] = cs_atom.tolist()
                result_dict[atom + "_mask"][key] = cs_atom_mask.tolist()
                result_dict['offset'][key] = offset_collect

            processed += 1

with open(os.path.join(base_path, "bmrb_cs_data_withoffset_lacs.json"), 'w') as f:
    json.dump(result_dict, f, indent=4)

print(f"Processed {processed} entries, results saved to bmrb_cs_data_withoffset_lacs.json")
print(f"There are {len(mismatch)} entries with residue mismatches:", mismatch)
print(f"There are {len(notfit_refer)} entries with no HA after rereferencing:", notfit_refer)

In [ ]:
# Remove mismatch entries from the saved JSON
import json, os

json_path = os.path.join(base_path, "bmrb_cs_data_withoffset_lacs.json")
new_json_path = os.path.join(base_path, "bmrb_cs_data_withoffset_lacs_v1.json")
with open(json_path, 'r') as f:
    data = json.load(f)

mismatch = ['bmr52649_1', 'bmr19825_1', 'bmr18265_1', 'bmr27587_1', 'bmr26648_1']

# Find numeric keys whose entry_ID is in the mismatch set
mismatch_keys = {k for k, v in data["entry_ID"].items() if v in mismatch}
print(f"Removing {len(mismatch_keys)} mismatch entries: {[data['entry_ID'][k] for k in mismatch_keys]}")

# Remove those keys from every field
for field in data:
    for k in mismatch_keys:
        data[field].pop(k, None)

with open(new_json_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Done. {len(data['entry_ID'])} entries remaining in {new_json_path}")